### Import modules and utility code

In [1]:
import flatdict as fd
import numpy as np
import yaml
import pint
from pint import UnitRegistry
ureg = UnitRegistry()
# ureg.formatter.default_format = "D"
# https://pint.readthedocs.io/en/stable/user/formatting.html

In [2]:
def recursive_assign(path, idx, leaf_idx, obj, nested_dict):
    if -1 < idx < leaf_idx:
        if path[idx] not in nested_dict:
            # print(f"\tgrp {level}")
            # nested_dict[f"{'/'.join(path[0:idx])}"] = {}
            nested_dict[path[idx]] = {}
        recursive_assign(path, idx + 1, leaf_idx, obj, nested_dict[path[idx]])
    else:
        if path[idx] not in nested_dict:
            # print(f"\t\tdset {level}")
            if isinstance(obj, ureg.Quantity):  # TODO fix
                if path[idx] not in nested_dict:
                    nested_dict[path[idx]] = {}
                    if obj.dimensionless:
                        if obj.units == "radian":
                            if np.shape(obj.magnitude) == ():
                                nested_dict[path[idx]]["value"] = obj.magnitude.item()
                            else:
                                nested_dict[path[idx]]["value"] = obj.magnitude.tolist()
                            nested_dict[path[idx]]["units"] = f"{obj.units}"
                        else:
                            if np.shape(obj.magnitude) == ():
                                nested_dict[path[idx]] = obj.magnitude.item()
                            else:
                                nested_dict[path[idx]] = obj.magnitude.tolist()
                    else:
                        if np.shape(obj.magnitude) == ():
                            nested_dict[path[idx]]["value"] = obj.magnitude.item()
                        else:
                            nested_dict[path[idx]]["value"] = obj.magnitude.tolist()
                        nested_dict[path[idx]]["units"] = f"{obj.units}"
            elif isinstance(obj, (str, bool)):
                if path[idx] not in nested_dict:
                    nested_dict[path[idx]] = obj
    return nested_dict

### Create a configuration file for SCORE following the NXmicrostructure_score_config.nxdl.xml contributed application definition

<div class="alert alert-block alert-info">Modify those values in this populated template that match your case.</div>

In [3]:
simid = 1
class ScoreConfig():
    def __init__(self):
        self.cfg: Dict[str, Any] = {}
        # physics
        self.material()
        self.deformation()
        self.nucleation()
        self.mobility()
        self.drag()
        self.recovery()
        self.time_temperature()
        # numerics
        self.discretization()
        self.numerics()
        self.solitary_unit_model()

        self.write_config_file()

    def discretization(self):
        """Configure microstructure instantiation."""
        self.cfg[f"entry1/discretization/NX_class"] = "NXmicrostructure"
        trg = "entry1/discretization/grid/"
        self.cfg[f"{trg}NX_class"] = "NXcg_grid"
        self.cfg[f"{trg}extent"] = ureg.Quantity(np.asarray([10, 10, 10], np.uint32))
        self.cfg[f"{trg}cell_dimensions"] = ureg.Quantity(np.float64(1.), ureg.micrometer)
   
    def deformation(self):
        """Configure deformed (initial) microstructure."""
        trg = "entry1/deformation/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}model"] = "ebsd"
        self.cfg[f"{trg}extent"] = ureg.Quantity(np.asarray([10., 10., 10.], np.float64), ureg.micrometer)
        self.cfg[f"{trg}diameter"] = ureg.Quantity(np.float64(10.), ureg.micrometer)
        # self.cfg[f"{trg}bunge_euler"] = ureg.Quantity(np.asarray([[]], np.float32, ureg.radians)
        self.cfg[f"entry1/deformation/ebsd/NX_class"] = "NXserialized"
        self.cfg[f"{trg}ebsd/path"] = "EBSDMap.osc"
        # TODO kwargs(cell_size, grain_size, grain_diameter, grain_euler, ebsd_filename)):

    def nucleation(self):
        """Configure nucleation model."""
        trg = "entry1/nucleation/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}spatial_distribution"] = "csr"
        self.cfg[f"{trg}incubation_time"] = "site_saturation"
        self.cfg[f"{trg}orientation"] = "sample_from_nucleus_euler"
        # self.cfg[f"{trg}bunge_euler"] = ureg.Quantity(np.asarray([[]], np.float32, ureg.radians)
        
    def material(self):
        """Configure material model single-phase only"""
        trg = "entry1/material_properties/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}reference_shear_modulus"] = ureg.Quantity(np.float64(9.5e9), ureg.pascal)
        self.cfg[f"{trg}reference_burgers_magnitude"] = ureg.Quantity(np.float64(0.286), ureg.nanometer)
        self.cfg[f"{trg}melting_temperature"] = ureg.Quantity(np.float64(800.), ureg.degC)

    def mobility(self):
        """Configure grain boundary mobility model."""
        model = "rollett_holm"  # "sebald_gottstein"  # "rollett_holm"
        if model == "sebald_gottstein":
            trg = "entry1/grain_boundary_mobility/"
            self.cfg[f"{trg}NX_class"] = "NXobject"
            self.cfg[f"{trg}model"] = "sebald_gottstein"
            trg += "sebald_gottstein/"
            self.cfg[f"{trg}NX_class"] = "NXobject"
            self.cfg[f"{trg}lagb_to_hagb_cut"] = ureg.Quantity(np.float64(15. / 180. * np.pi), ureg.radians)
            self.cfg[f"{trg}lagb_pre_factor"] = ureg.Quantity(np.float64(300.), ureg.meter ** 4 / (ureg.joule * ureg.second))
            self.cfg[f"{trg}lagb_enthalpy"] = ureg.Quantity(np.float64(2.45), ureg.eV)
            self.cfg[f"{trg}hagb_pre_factor"] = ureg.Quantity(np.float64(300.), ureg.meter ** 4 / (ureg.joule * ureg.second))
            self.cfg[f"{trg}hagb_enthalpy"] = ureg.Quantity(np.float64(2.45), ureg.eV)
            self.cfg[f"{trg}special_disorientation"] = ureg.Quantity(np.float64(10. / 180. * np.pi), ureg.radians)
            self.cfg[f"{trg}special_pre_factor"] = ureg.Quantity(np.float64(300.), ureg.meter ** 4 / (ureg.joule * ureg.second))
            self.cfg[f"{trg}special_enthalpy"] = ureg.Quantity(np.float64(2.45), ureg.eV)
            # MAXDISORI_TO_40DEG111
        elif model == "rollett_holm":
            trg = "entry1/grain_boundary_mobility/"
            self.cfg[f"{trg}model"] = "rollett_holm"
            trg += "rollett_holm/"
            self.cfg[f"{trg}NX_class"] = "NXobject"
            self.cfg[f"{trg}m_null"] = ureg.Quantity(np.float64(300.), ureg.meter ** 4 / (ureg.joule * ureg.second))
            self.cfg[f"{trg}enthalpy"] = ureg.Quantity(np.float64(1.), ureg.eV)
            self.cfg[f"{trg}c1"] = ureg.Quantity(np.float64(0.9))
            self.cfg[f"{trg}c2"] = ureg.Quantity(np.float64(5.))
            self.cfg[f"{trg}c3"] = ureg.Quantity(np.float64(9.))

    def recovery(self):
        """Configure stored elastic energy recovery model."""
        trg = "entry1/stored_energy_recovery_model/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}model"] = "none"

    def drag(self):
        """Configure reduction of grain boundary mobility by dispersoids (Zener-Smith drag)."""
        trg = "entry1/dispersoid_drag/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}model"] = "none"
        trg = "entry1/dispersoid_drag/zener_smith/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}pre_factor"] = ureg.Quantity(np.float64(1.5))
        self.cfg[f"{trg}surface_energy"] = ureg.Quantity(np.float64(0.324), ureg.joule / (ureg.meter ** 2))
        self.cfg[f"{trg}time"] = ureg.Quantity(np.asarray([0., 3600.], np.float64), ureg.second)
        self.cfg[f"{trg}radius"] = ureg.Quantity(np.asarray([0., 0.], np.float64), ureg.nanometer)

    def time_temperature(self):
        """Configure the simulated annealing schedule."""
        trg = "entry1/time_temperature/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}time"] = ureg.Quantity(np.asarray(
    [0., 1., 2., 3., 4., 5., 6., 7., 8., 9., 10.,
    11., 12., 16., 20., 21., 22., 23., 24., 25.,
    26., 27., 28., 29., 30., 31., 32., 36., 40.,
    41., 42., 43., 44., 45., 46., 47., 48., 49.,
    50., 51., 52., 53., 54., 55., 56., 57., 58.,
    3600.], np.float64), ureg.second)
        self.cfg[f"{trg}temperature"] = ureg.Quantity(np.asarray(
    [20., 165., 310., 455., 600., 600., 600., 600., 600., 455., 310., 165., 20., 20., 20., 165., 
            310., 455., 600., 600., 600., 600., 600., 455., 310., 165., 20., 20., 20., 165., 310., 455., 
            600., 600., 600., 600., 600., 600., 600., 600., 600., 600., 600., 455., 310., 165., 20., 20.], np.float64), ureg.degC)

    # ideal components

    def numerics(self):
        """Configure stop criteria and numerics."""
        trg = "entry1/numerics/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}max_delta_x"] = ureg.Quantity(np.float64(0.1))
        self.cfg[f"{trg}initial_cell_cache"] = ureg.Quantity(np.float64(0.2))
        self.cfg[f"{trg}realloc_cell_cache"] = ureg.Quantity(np.float64(1.))
        self.cfg[f"{trg}defragm_cell_cache"] = True
        self.cfg[f"{trg}defragm_x"] = ureg.Quantity(np.asarray([0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90,
                      0.92, 0.94, 0.96, 0.98, 0.99, 0.999, 0.9999], np.float64))
        self.cfg["entry1/sampling/x"] = ureg.Quantity(np.asarray(np.linspace(0., stop=1., num=241, endpoint=True), np.float64))
        # self.cfg[f"{trg}percolation"] = False

        self.cfg[f"{trg}max_x"] = ureg.Quantity(np.float64(1.))
        self.cfg[f"{trg}max_time"] = ureg.Quantity(np.float64(3600.), ureg.second)
        self.cfg[f"{trg}max_iteration"] = ureg.Quantity(np.uint32(10000))

    def solitary_unit_model(self):
        """Configure solitary unit post-processing approach."""
        trg = "entry1/solitary_unit/"
        self.cfg[f"{trg}NX_class"] = "NXobject"
        self.cfg[f"{trg}apply"] = False
        self.cfg[f"{trg}number_of_domains"] = ureg.Quantity(np.uint32(1))
        self.cfg[f"{trg}rediscretization"] = ureg.Quantity(np.uint32(10000))
        # self.cfg[f"{trg}domain_identifier"] = None

    def write_config_file(self):
        """Write SCORE YAML config file."""
        template = {
            "NX_class": "NXroot",
            # "file_time": None,
            "NeXus_version": None,
            # "HDF5_version": None,
            # "h5py_version": None,
            "file_update_time": None,
            "entry1": {
                "NX_class": "NXentry",
                "definition": "NXmicrostructure_score_config",
                "simulation_identifier": None,
                "start_time": None,
                # "end_time": None,
                "program1": None
            }
        }
        # TODO add YAML header        
        for key, obj in self.cfg.items():      
            path = key.split("/")
            recursive_assign(path, 0, len(path) - 1, obj, template)

        with open(f"NXmicrostructure_score_config.SimID.{simid}.yaml", "w") as fp:
            yaml.dump(template, fp, default_flow_style=None, sort_keys=False)

In [4]:
yml = ScoreConfig()
yml.write_config_file()

### Display a flattened list of all NeXus/HDF5 instance names

In [6]:
locs = set()
for example_filename in [f"NXmicrostructure_score_config.SimID.{simid}.yaml"]:
    with open(example_filename, "r", encoding="utf-8") as stream:
        flat_mdata = fd.FlatDict(yaml.safe_load(stream), delimiter="/")
        for key, val in flat_mdata.items():
            # print(f"key: {key}, value: {val}")
            if not key.endswith("NX_class"):
                loc = "root"
                print(key)
                for token in key.split("/"):
                    loc += f'''["{token}"]'''
                loc += " >>;"
                locs.add(loc)
# for loc in sorted(locs):
#     print(loc)

NeXus_version
file_update_time
entry1/definition
entry1/simulation_identifier
entry1/start_time
entry1/program1
entry1/material_properties/reference_shear_modulus/value
entry1/material_properties/reference_shear_modulus/units
entry1/material_properties/reference_burgers_magnitude/value
entry1/material_properties/reference_burgers_magnitude/units
entry1/material_properties/melting_temperature/value
entry1/material_properties/melting_temperature/units
entry1/deformation/model
entry1/deformation/extent/value
entry1/deformation/extent/units
entry1/deformation/diameter/value
entry1/deformation/diameter/units
entry1/deformation/ebsd/path
entry1/nucleation/spatial_distribution
entry1/nucleation/incubation_time
entry1/nucleation/orientation
entry1/grain_boundary_mobility/model
entry1/grain_boundary_mobility/rollett_holm/m_null/value
entry1/grain_boundary_mobility/rollett_holm/m_null/units
entry1/grain_boundary_mobility/rollett_holm/enthalpy/value
entry1/grain_boundary_mobility/rollett_holm/ent

***

In [ ]:
# only debug code up from this point
# customizations for Zeiss
ureg.define("mobility = 1 * m ** 4 / (J * s)")
a = ureg.Quantity("1. * m ** 4 / (J * s)")
print(a)
b = ureg.Quantity(np.float64(10.))
print(np.shape(b.magnitude) == ())
b = ureg.Quantity(np.pi, ureg.radian)
print(b.dimensionless)
print(b.units == "radian")
# TODO define custom ureg for eV
# TODO define custom ureg for ureg.mobility ureg.meter ** 4 / (ureg.joule * ureg.second)